In [0]:
from pyspark.sql import functions as F

# 1. Read parameters from ADF widgets
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type").strip().lower()

print("===== Parameters =====")
print(f"landing_timestamp : {landing_timestamp}")
print(f"table_name        : {table_name}")
print(f"load_type         : {load_type}")
print(f"merge_keys        : {merge_keys}")



In [0]:
bronze_table = f"adb_hospital.bronze.{table_name}"
silver_table = f"adb_hospital.silver.{table_name}"
source_view = f"vw_{table_name}_src"

print(f"Bronze table : {bronze_table}")
print(f"Silver table : {silver_table}")
print(f"Source view  : {source_view} (used only for merge)")

In [0]:
bronze_df = (
    spark.read.table(bronze_table)
    .filter(F.col("landing_timestamp") == landing_timestamp)
)

print(f"Reading from {bronze_table} where landing_timestamp = '{landing_timestamp}'")
print(f"Bronze records found: {bronze_df.count()}")

In [0]:
silver_df = (
    bronze_df
    .withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)

print("Added audit columns: insert_timestamp, update_timestamp")
print(f"Silver DataFrame columns: {silver_df.columns}")

In [0]:
print(f"Starting {load_type.upper()} load into {silver_table}")

if load_type == "full":

    # Replace Silver table
    print("FULL load: overwriting Silver table...")
    (silver_df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
    )
    print(f"FULL load complete: {silver_table} overwritten")


elif load_type == "append":

    # Append records to Silver
    print("APPEND load: appending records to Silver table...")
    (silver_df.write
        .mode("append")
        .saveAsTable(silver_table)
    )
    print(f"APPEND load complete: records appended to {silver_table}")


elif load_type == "merge":

    # Read merge keys
    keys = [key.strip() for key in merge_keys.split(",") if key.strip()]

    if not keys:
        raise ValueError("Merge keys are required for MERGE load")
    print(f"MERGE load: using merge keys {keys}")

    # Remove duplicate records
    source_df = silver_df.dropDuplicates(keys)
    print(f"Source records after removing duplicates on keys: {source_df.count()}")

    # Create temporary view
    source_df.createOrReplaceTempView(source_view)
    print(f"Temp view created: {source_view}")

    if not spark.catalog.tableExists(silver_table):

        # Create Silver table on first run
        print(f"{silver_table} does not exist: creating it from {source_view}...")
        spark.sql(f"""
            CREATE TABLE {silver_table}
            AS SELECT * FROM {source_view}
        """)
        print(f"Silver table created: {silver_table}")

    else:

        # Build merge condition
        condition = " AND ".join(
            [f"t.`{key}` = s.`{key}`" for key in keys]
        )

        # Update all columns except insert_timestamp
        update_columns = ", ".join(
            [f"t.`{col}` = s.`{col}`"
             for col in source_df.columns
             if col != "insert_timestamp"]
        )

        # Execute MERGE
        print(f"Merge condition: {condition}")
        print(f"Running MERGE INTO {silver_table}...")
        merge_result = spark.sql(f"""
            MERGE INTO {silver_table} t
            USING {source_view} s
            ON {condition}

            WHEN MATCHED THEN
                UPDATE SET {update_columns}

            WHEN NOT MATCHED THEN
                INSERT *
        """)
        print("MERGE complete. Result metrics:")
        display(merge_result)


else:
    raise ValueError(
        f"Unsupported load type: {load_type}. "
        "Expected FULL, APPEND, or MERGE."
    )


In [0]:
record_count = silver_df.count()

print(f"Records processed: {record_count}")
print(f"Silver table total rows: {spark.read.table(silver_table).count()}")
print("Notebook finished, returning record count to caller")

dbutils.notebook.exit(str(record_count))